<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/14_multi_asset_expansion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
from src.patterns.scanner import scan_all_patterns
from src.labeling.label_events import label_events
from src.features.build_features import build_feature_matrix

def process_ticker(ticker, start='2010-01-01', end='2026-01-01'):
    """Download and run full detection+labeling pipeline on one ticker."""
    raw = yf.download(ticker, start=start, end=end, auto_adjust=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']]
    if raw.index.tz is not None:
        raw.index = raw.index.tz_localize(None)
    raw.index.name = 'Date'

    scanned = scan_all_patterns(raw.copy())
    EXCLUDE = ['triangle_pattern', 'channel_pattern']  # match your grid-search ablation convention if desired, or remove this line to include everything
    features, labels, labeled_df = build_feature_matrix(raw.copy())

    return {
        'ticker': ticker, 'raw': raw, 'scanned': scanned,
        'features': features, 'labels': labels, 'labeled_df': labeled_df,
    }

# 9 candidate instruments: mix of index ETFs, large-cap tech, and one non-tech
# large-cap for diversity, per your supervisor's market-cap/volume suggestion
TICKERS = ['QQQ', 'NVDA', 'GOOGL', 'AAPL', 'MSFT', 'AMZN', 'META', 'JPM', 'XOM']

results_by_ticker = {}
for t in TICKERS:
    try:
        results_by_ticker[t] = process_ticker(t)
        print(f"{t}: OK — {results_by_ticker[t]['scanned']['has_event'].sum()} events, "
              f"{results_by_ticker[t]['features'].shape[0]} labeled")
    except Exception as e:
        print(f"{t}: FAILED — {e}")

# Summary table
summary = pd.DataFrame([
    {
        'Ticker': t,
        'Bars': len(r['raw']),
        'Events': r['scanned']['has_event'].sum(),
        'Channels': r['scanned']['channel_pattern'].notna().sum(),
        'Triangles': r['scanned']['triangle_pattern'].notna().sum(),
        'Long %': (r['labels'] == 'long').mean() * 100,
        'Short %': (r['labels'] == 'short').mean() * 100,
        'No-trade %': (r['labels'] == 'no_trade').mean() * 100,
    }
    for t, r in results_by_ticker.items()
])
print(summary.to_string(index=False))

ModuleNotFoundError: No module named 'src'